# Exercise: Creative Writing Assistant
## AIAT 124 - Generative AI

## Learning Objectives

- Generate creative text with a language model
- Fine-tune a pre-trained model on your own writing style
- Control generation with sampling parameters (temperature, top-k)
- Evaluate generation quality with computed metrics

## Real-World Context

Build an AI writing assistant for content creation and storytelling.

**Task**: Generate creative stories and articles.

---

## Task 1: Text Generation Setup and Sampling Control (25 points)

The setup cell below trains a small character-level LSTM (our local stand-in for a pre-trained
model — same idea as loading GPT-2, without the download) and includes a **complete, working**
`generate()` with temperature sampling. It is given to you as a reference — study it, then earn
the points:

1. **Run** the cell and compare the low- vs high-temperature samples it prints
2. **Explain** (in a comment or markdown cell you add) *why* temperature changes the output —
   trace it through `logits / temperature → softmax → sample`
3. **Extend** `generate()` with a `top_k` parameter that keeps only the `k` most likely
   characters before sampling — a `TODO` marker at the end of the cell shows where and how

## 📥 Inputs & 📤 Outputs

**Inputs:** What we use in this notebook

- Libraries and concepts as introduced in this notebook; see prerequisites and code comments.

**Outputs:** What you'll see when you run the cells

- Printed results, figures, and summaries as shown when you run the cells.

---

In [1]:
# ── Setup: pre-trained character LSTM language model ─────────────────────────
# (In real projects you'd load GPT-2 etc.; we use a local LSTM to avoid downloads.)
import torch, torch.nn as nn, torch.optim as optim, numpy as np
torch.manual_seed(0); np.random.seed(0)

corpus = ('the quick brown fox jumps over the lazy dog. ' * 20 +
          'hello world this is a test of the language model. ' * 20)
chars = sorted(set(corpus)); c2i={c:i for i,c in enumerate(chars)}
i2c={i:c for c,i in c2i.items()}; V,S=len(chars),30

X=torch.tensor([[c2i[corpus[j+k]] for k in range(S)] for j in range(len(corpus)-S)],dtype=torch.long)
y=torch.tensor([c2i[corpus[j+S]] for j in range(len(corpus)-S)],dtype=torch.long)

class CharLM(nn.Module):
    def __init__(self): super().__init__(); self.e=nn.Embedding(V,32); self.l=nn.LSTM(32,128,batch_first=True); self.f=nn.Linear(128,V)
    def forward(self,x): return self.f(self.l(self.e(x))[0][:,-1,:])

model=CharLM(); opt=optim.Adam(model.parameters(),lr=3e-3); crit=nn.CrossEntropyLoss()
from torch.utils.data import TensorDataset,DataLoader
loader=DataLoader(TensorDataset(X,y),batch_size=64,shuffle=True)
print('Training model...')
for ep in range(10):
    model.train(); el=0
    for xb,yb in loader:
        opt.zero_grad(); loss=crit(model(xb),yb); loss.backward(); opt.step(); el+=loss.item()
print(f'Done. Final loss={el/len(loader):.4f}')

# ── Task 1: Implement the generate() function ─────────────────────────────
# Fill in the body: given a `seed` string, generate `n` new characters.
# Hint: use temperature sampling: logits/temperature → softmax → np.random.choice
def generate(seed: str, n: int = 50, temperature: float = 1.0) -> str:
    model.eval()
    out = seed
    ctx = [c2i.get(c, 0) for c in seed[-S:]]
    for _ in range(n):
        x = torch.tensor([ctx], dtype=torch.long)
        with torch.no_grad():
            logits = model(x)[0] / max(temperature, 1e-6)
        probs = torch.softmax(logits, 0).numpy()
        nxt = int(np.random.choice(V, p=probs))
        out += i2c[nxt]
        ctx = ctx[1:] + [nxt]
    return out

# ── Task 2: Test your generate() with two temperature settings ─────────────
print('\nLow temperature (deterministic):')
print(generate('the quick', n=50, temperature=0.3))
print('\nHigh temperature (creative):')
print(generate('the quick', n=50, temperature=1.5))

# ── TODO (Task 1.3): add a top_k parameter to generate() ────────────────────
# Goal: generate(seed, n, temperature, top_k=3) samples only among the 3 most likely
# next characters. Hint: before softmax, set every logit that is NOT among the k
# largest to -float('inf') (see torch.topk). Then compare top_k=3 vs full sampling.

Training model...


Done. Final loss=0.0106

Low temperature (deterministic):
the quick brown fox jumps over the lazy dog. the lazy dog. 

High temperature (creative):
the quick buela helc og. the tizs a of the lazy mof. hel o 


## Task 2: Fine-tuning on Your Writing Style (35 points)

**Fine-tuning** means *continuing training* of an already-trained model on a smaller,
specialized dataset — the model keeps its general knowledge and adapts to the new style.
You already have the pre-trained model from Task 1, so everything happens right here
(the Hugging Face `Trainer` workflow does the same thing at scale — same idea, bigger model).

Your work for the points:

1. **Replace** the sample corpus in the next cell with **your own** creative-writing text
   (≥ 500 characters; lowercase letters, spaces, and periods — the model's alphabet)
2. **Fine-tune** using the provided loop — note the *lower learning rate* and *few epochs*,
   and explain in a comment why they protect the pre-trained knowledge
3. **Compare** generations before vs after fine-tuning (same seed, same temperature) and
   describe the style change in 2-3 sentences

In [2]:
# Task 2 — fine-tune the pre-trained model from Task 1 on a creative-writing corpus.
# The scaffold runs end-to-end on a sample corpus; your graded work is swapping in
# YOUR corpus and analyzing the before/after change.

# ── Sample creative corpus (REPLACE with >= 500 chars of your own writing) ──
style_corpus = ("the dragon flew over the misty mountain and the wizard spoke a quiet spell. "
                "a hero walked into the dark forest as cold stars watched the silent road. "
                "the old king dreamed of a silver ship sailing beyond the edge of the world. ") * 6

# ── Keep only characters the pre-trained model knows (its vocab is FIXED) ──
# A real fine-tune reuses the pre-trained tokenizer for the same reason.
ft_text = ''.join(c for c in style_corpus.lower() if c in c2i)
print(f'Fine-tuning corpus: {len(ft_text)} usable characters')

# ── Build training windows exactly like the setup cell did ────────────────
Xf = torch.tensor([[c2i[ft_text[j+k]] for k in range(S)] for j in range(len(ft_text)-S)], dtype=torch.long)
yf = torch.tensor([c2i[ft_text[j+S]] for j in range(len(ft_text)-S)], dtype=torch.long)
ft_loader = DataLoader(TensorDataset(Xf, yf), batch_size=64, shuffle=True)

# ── BEFORE snapshot: what the pre-trained model says to a fantasy seed ────
print('\nBefore fine-tuning:', generate('the dragon flew', n=60, temperature=0.5))

# ── Fine-tune: same loop as pre-training but LOWER lr and FEW epochs ──────
# (a high lr or many epochs would overwrite — "catastrophically forget" — Task 1's training)
opt_ft = optim.Adam(model.parameters(), lr=5e-4)
for ep in range(8):
    model.train(); el = 0
    for xb, yb in ft_loader:
        opt_ft.zero_grad(); loss = crit(model(xb), yb); loss.backward(); opt_ft.step(); el += loss.item()
print(f'\nFine-tuned 8 epochs. Final loss = {el/len(ft_loader):.4f}')

# ── AFTER snapshot: same seed, same temperature — only the training changed ─
print('After fine-tuning :', generate('the dragon flew', n=60, temperature=0.5))

# TODO (Task 2.1): replace style_corpus above with your own text and re-run.
# TODO (Task 2.3): in a comment here, describe how the output style changed and why.

Fine-tuning corpus: 1356 usable characters

Before fine-tuning: the dragon flew. hello world this is a test of the lazy dog. the quick brow



Fine-tuned 8 epochs. Final loss = 1.1015
After fine-tuning : the dragon flew orllo wron fangumpe overld this isa tangua deld tis isa tan


## Task 3: Generation and Evaluation (40 points)

Use the cell below to generate at three temperatures and compute a simple quality metric
(**distinct 4-gram ratio** — near 0 means the text loops and repeats, near 1 means every
4-character window is unique, which usually reads as noise).

Your work for the points:

1. **Run** the evaluation and read the metric alongside the samples
2. **Interpret** the trade-off the numbers show (repetition ↔ incoherence) and pick the
   temperature you would ship in a writing assistant, justifying your choice
3. **Add one metric of your own** (ideas: fraction of tokens that are real English words,
   average sentence length, share of characters inside quoted dialogue)

**Submission**: Completed notebook — your corpus in Task 2, your `top_k` extension in Task 1,
your metric and written interpretation here.

**Grading**: 100 points total (Task 1: 25, Task 2: 35, Task 3: 40)

In [3]:
# Task 3 — generate at three temperatures and COMPUTE a quality metric for each.
# The numbers come from the model's actual output — interpret them, don't assume them.
def distinct_ngram_ratio(s, n=4):
    """Unique n-grams / total n-grams: ~0 = repetitive loops, ~1 = maximal novelty."""
    grams = [s[i:i+n] for i in range(len(s) - n + 1)]
    return len(set(grams)) / max(len(grams), 1)

seed = 'the dragon flew'
print(f"{'temp':>5s} {'distinct-4':>10s}  sample (first 60 chars)")
for temp in [0.3, 0.8, 1.5]:
    out = generate(seed, n=200, temperature=temp)
    ratio = distinct_ngram_ratio(out)
    print(f"{temp:5.1f} {ratio:10.2f}  {out[:60]!r}")

# TODO (Task 3.2): which temperature would you ship, and why? Answer in a comment.
# TODO (Task 3.3): implement one more metric and add a column for it.

 temp distinct-4  sample (first 60 chars)
  0.3       0.41  'the dragon flew orld this isa tangua the modld the azy dog. '
  0.8       0.84  'the dragon flewd ofrdis barmpste the madeovt msell. her olqk'
  1.5       0.98  'the dragon flew orox rolc dnisv atjtsw eaniuage hoad hpe. di'
